# КЛАССИФИКАЦИЯ — ОПТИМИЗАЦИЯ ГИПЕРПАРАМЕТРОВ

## Цель
Для каждой из 4 задач взять топ-2 модели из baseline и оптимизировать их гиперпараметры с помощью RandomizedSearchCV.

## План
1. Загрузка подготовленных данных и топ-моделей
2. Оптимизация гиперпараметров для каждой задачи
3. Сравнение результатов с baseline
4. Сохранение лучших моделей

## Параметры оптимизации
- Метод: RandomizedSearchCV
- Количество итераций: 20
- Кросс-валидация: 5-fold
- Метрика: F1 (максимизация)
- Random state: 42

In [10]:
import sys
import os
import pandas as pd
import numpy as np
import joblib

sys.path.append(os.path.dirname(os.getcwd()))

from src import get_param_grids
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

## 1. Загрузка данных и моделей

**Загружаемые данные:**
- `X_train_scaled.pkl` / `X_test_scaled.pkl` — признаки (масштабированные)
- `y_train_clf.pkl` / `y_test_clf.pkl` — бинарные метки для 4 задач

**Загружаемые артефакты:**
- `top_models_classification.pkl` — топ-2 модели для каждой задачи (из ноутбука 04)
- `param_grids` — сетки гиперпараметров для каждой модели (из src.models)

In [11]:
SAVE_PATH = '../data/processed/'

X_train = joblib.load(f'{SAVE_PATH}X_train_scaled.pkl')
X_test = joblib.load(f'{SAVE_PATH}X_test_scaled.pkl')
y_train_clf = joblib.load(f'{SAVE_PATH}y_train_clf.pkl')
y_test_clf = joblib.load(f'{SAVE_PATH}y_test_clf.pkl')

# Загружаем топ-модели
top_models_clf = joblib.load('../artifacts/top_models_classification.pkl')
param_grids = get_param_grids()

CLASSIFICATION_TASKS = {
    'IC50_binary': 'IC50 > медианы',
    'CC50_binary': 'CC50 > медианы',
    'SI_binary': 'SI > медианы',
    'SI_8_binary': 'SI > 8'
}

## 2. Оптимизация гиперпараметров

Для каждой задачи:
1. Берём топ-2 модели из baseline
2. Проверяем наличие сетки параметров
3. Запускаем RandomizedSearchCV для каждой модели
4. Выбираем модель с лучшим F1 на тестовой выборке
5. Сохраняем лучшую модель

**Важно:** Для DecisionTree может не быть сетки параметров, поэтому он будет пропущен.

In [18]:
# Структура для результатов
best_models = {}
tuned_results = {task: {} for task in CLASSIFICATION_TASKS.keys()}


for task_col, task_name in CLASSIFICATION_TASKS.items():

    print(f" {task_name} ({task_col})")
    print('─'*60)
    
    y_train = y_train_clf[task_col]
    y_test = y_test_clf[task_col]
    
    # Топ-2 модели для этой задачи
    top_2 = top_models_clf[task_col]
    
    baseline_df = pd.read_csv(f'../artifacts/results/classification_baseline_{task_col}.csv', index_col=0)
    best_score = -np.inf
    best_model = None
    
    for model_name in top_2:
        print(f" Оптимизация: {model_name}")
        
        # Создаём модель
        if 'RandomForest' in model_name:
            from sklearn.ensemble import RandomForestClassifier
            base_model = RandomForestClassifier(random_state=42)
        elif 'XGBoost' in model_name:
            from xgboost import XGBClassifier
            base_model = XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False)
        elif 'GradientBoosting' in model_name:
            from sklearn.ensemble import GradientBoostingClassifier
            base_model = GradientBoostingClassifier(random_state=42)
        elif 'DecisionTree' in model_name:
            from sklearn.tree import DecisionTreeClassifier
            base_model = DecisionTreeClassifier(random_state=42)
        else:
            continue
        
        # Подбираем параметры
        rs = RandomizedSearchCV(
            base_model,
            param_grids.get(model_name, {}),
            n_iter=20,
            cv=5,
            scoring='f1',
            random_state=42,
            n_jobs=-1,
            verbose=0
        )
        rs.fit(X_train, y_train)
        
        print(f"    Лучшие параметры: {rs.best_params_}")
        y_pred = rs.predict(X_test)
        test_f1 = f1_score(y_test, y_pred)
        print(f"    CV F1: {rs.best_score_:.4f}    Test F1: {test_f1:.4f}")
        baseline_f1 = baseline_df.loc[model_name, 'F1']
       
        
        # Сохраняем результаты
        tuned_results[task_col][model_name] = {
            'F1': test_f1,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred),
            'Recall': recall_score(y_test, y_pred),
            'best_params': rs.best_params_
        }
        
        # Выбираем лучшую
        if baseline_f1 > test_f1:
            # Используем baseline модель (нужно создать заново)
            if 'RandomForest' in model_name:
                best_for_model = RandomForestClassifier(random_state=42)
            elif 'XGBoost' in model_name:
                best_for_model = XGBClassifier(random_state=42, verbosity=0)
            elif 'GradientBoosting' in model_name:
                best_for_model = GradientBoostingClassifier(random_state=42)
            elif 'DecisionTree' in model_name:
                best_for_model = DecisionTreeClassifier(random_state=42)    
            best_for_model.fit(X_train, y_train)
            
        if baseline_f1 > best_score:
                best_score = baseline_f1
                best_model = best_for_model

        else:
            if test_f1 > best_score:
                best_score = test_f1
                best_model = rs.best_estimator_
    
    # Сохраняем лучшую модель
    best_models[task_col] = best_model
    joblib.dump(best_model, f'../artifacts/models/best_{task_col}.pkl')
    
    print(f"\nЛучшая модель для {task_name}: {best_model.__class__.__name__}   Test F1: {best_score:.4f}")

 IC50 > медианы (IC50_binary)
────────────────────────────────────────────────────────────
 Оптимизация: GradientBoosting


    Лучшие параметры: {'n_estimators': 50, 'min_samples_split': 2, 'max_depth': 3, 'learning_rate': 0.01}
    CV F1: 0.7431    Test F1: 0.6820
 Оптимизация: RandomForest
    Лучшие параметры: {'n_estimators': 50, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 5}
    CV F1: 0.7374    Test F1: 0.6866

Лучшая модель для IC50 > медианы: GradientBoostingClassifier   Test F1: 0.7251
 CC50 > медианы (CC50_binary)
────────────────────────────────────────────────────────────
 Оптимизация: XGBoost
    Лучшие параметры: {'subsample': 1.0, 'n_estimators': 100, 'max_depth': 9, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
    CV F1: 0.7497    Test F1: 0.8020
 Оптимизация: GradientBoosting
    Лучшие параметры: {'n_estimators': 50, 'min_samples_split': 10, 'max_depth': 3, 'learning_rate': 0.05}
    CV F1: 0.7441    Test F1: 0.7778

Лучшая модель для CC50 > медианы: RandomForestClassifier   Test F1: 0.7852
 SI > медианы (SI_binary)
───────────────────────────────────────────────────────

In [19]:
for task_col, task_name in CLASSIFICATION_TASKS.items():
    print(f"{task_name} — Результаты оптимизации")
    print('─'*60)
    
    df = pd.DataFrame(tuned_results[task_col]).T
    display(df.round(4))
    df.to_csv(f'../artifacts/results/classification_tuned_{task_col}.csv')

IC50 > медианы — Результаты оптимизации
────────────────────────────────────────────────────────────


,F1,Accuracy,Precision,Recall,best_params
GradientBoosting,0.682028,0.655,0.606557,0.778947,"{'n_estimators': 50, 'min_samples_split': 2, '..."
RandomForest,0.686567,0.685,0.650943,0.726316,"{'n_estimators': 50, 'min_samples_split': 2, '..."


CC50 > медианы — Результаты оптимизации
────────────────────────────────────────────────────────────


,F1,Accuracy,Precision,Recall,best_params
XGBoost,0.80198,0.8,0.852632,0.757009,"{'subsample': 1.0, 'n_estimators': 100, 'max_d..."
GradientBoosting,0.777778,0.76,0.770642,0.785047,"{'n_estimators': 50, 'min_samples_split': 10, ..."


SI > медианы — Результаты оптимизации
────────────────────────────────────────────────────────────


,F1,Accuracy,Precision,Recall,best_params
RandomForest,0.65641,0.665,0.673684,0.64,"{'n_estimators': 50, 'min_samples_split': 10, ..."
DecisionTree,0.64,0.64,0.64,0.64,"{'min_samples_split': 20, 'min_samples_leaf': ..."


SI > 8 — Результаты оптимизации
────────────────────────────────────────────────────────────


,F1,Accuracy,Precision,Recall,best_params
GradientBoosting,0.564885,0.715,0.672727,0.486842,"{'n_estimators': 100, 'min_samples_split': 5, ..."
XGBoost,0.592593,0.725,0.677966,0.526316,"{'subsample': 0.8, 'n_estimators': 200, 'max_d..."


## Сравнение с baseline

Анализируем, удалось ли улучшить качество моделей.
- **Baseline F1** — качество модели до оптимизации
- **Tuned F1** — качество модели после оптимизации
- **Улучшение** = Tuned F1 - Baseline F1

In [20]:
comparison = {}

for task_col, task_name in CLASSIFICATION_TASKS.items():   
    # Загружаем baseline результаты
    baseline_df = pd.read_csv(f'../artifacts/results/classification_baseline_{task_col}.csv', index_col=0)
    
    # Топ-2 модели
    top_2 = top_models_clf[task_col]
    
    comparison[task_col] = {}
    for model_name in top_2:
        if model_name not in tuned_results[task_col]:
            continue
            
        baseline_f1 = baseline_df.loc[model_name, 'F1']
        tuned_f1 = tuned_results[task_col][model_name]['F1']
        improvement = tuned_f1 - baseline_f1
        
        comparison[task_col][model_name] = {
            'Baseline F1': baseline_f1,
            'Tuned F1': tuned_f1,
            'Improvement': improvement
        }
        
        print(f"\n{task_name} — {model_name}:")
        print(f"  Baseline F1:{baseline_f1:.4f}     Tuned F1:{tuned_f1:.4f}     Улучшение:{improvement:+.4f}")

# Сохраняем сравнение
os.makedirs('../artifacts/results/', exist_ok=True)
joblib.dump(comparison, '../artifacts/results/classification_comparison.pkl')


IC50 > медианы — GradientBoosting:
  Baseline F1:0.7251     Tuned F1:0.6820     Улучшение:-0.0431

IC50 > медианы — RandomForest:
  Baseline F1:0.7101     Tuned F1:0.6866     Улучшение:-0.0235

CC50 > медианы — XGBoost:
  Baseline F1:0.7852     Tuned F1:0.8020     Улучшение:+0.0168

CC50 > медианы — GradientBoosting:
  Baseline F1:0.7802     Tuned F1:0.7778     Улучшение:-0.0024

SI > медианы — RandomForest:
  Baseline F1:0.6700     Tuned F1:0.6564     Улучшение:-0.0136

SI > медианы — DecisionTree:
  Baseline F1:0.6599     Tuned F1:0.6400     Улучшение:-0.0199

SI > 8 — GradientBoosting:
  Baseline F1:0.7235     Tuned F1:0.5649     Улучшение:-0.1586

SI > 8 — XGBoost:
  Baseline F1:0.7235     Tuned F1:0.5926     Улучшение:-0.1309


['../artifacts/results/classification_comparison.pkl']

**Ключевые выводы:**

1. **Единственное значимое улучшение** — CC50 > медианы (XGBoost, F1 = 0.802)
2. **Остальные задачи** показали ухудшение или остались на уровне baseline
3. **SI > 8** — самая проблемная задача (F1 = 0.593, ухудшение на 0.131)
4. **RandomForest** — стабильный выбор для большинства задач

**Причины отсутствия улучшения:**
- Недостаточный объём выборки 
- Случайность валидационного сплита
- Сетка параметров не покрыла оптимальную область
- Переобучение на кросс-валидации

**Рекомендации:**
1. Увеличить число итераций RandomizedSearchCV до 100
2. Добавить сетку параметров для DecisionTree
3. Использовать GridSearchCV для тонкой настройки
4. Попробовать другие модели (LightGBM, CatBoost)